# 04c. Prediction-time evaluation (additional analysis)

This is separate from notebook 04b's original RAAS association analyses and retrospective-discrimination classifier. Read the [prespecified protocol](../docs/PREDICTION_TIME_PROTOCOL.md) before running. Admission and 24-hour models use different at-risk populations; their AUCs must not be presented as model improvement. The outcome is subsequent in-hospital death until discharge, not a fixed-day or post-discharge outcome.

Eligibility uses only ICU entry up to each prediction instant and retains later ICU transfers. The primary predictors are reconstructed age and recorded sex, assumed known at admission. A 24-hour companion adds only early prescription records linked to POE order times by that instant. The limited predictors and retrospective field/version-history limitations prevent a claim of deployment-ready or prospective validation. No identifiers, individual probabilities or raw inputs are exported.

Both risk sets require a confirmed target-ward transfer interval at the prediction instant. Missing/ambiguous location is excluded; later ICU transfers remain followed through hospital discharge. Admission results apply to the ward-confirmed subset only. See the documented ward-eligibility amendment in the protocol.


## 1. Imports and source configuration

## Reading the results

Data sources and execution mode are summarized once below, after the destination settings. Expand **Calculation inputs and source definitions** to trace individual SQL definitions and aggregate references.

Each result is generated by Python in this order: **number/title → figure or aggregate table → how to read → computed result summary → interpretation limits**. Cohort definitions, exposures, adjustment variables and estimators are unchanged. See the [reproduction instructions](../REPRODUCIBILITY.md#python-generated-notebook-results).


The input-definition index describes the Python objects/queries used in calculation. The separate **Saved aggregates and verification files (CSV/JSON)** list identifies saved results and checking records, rather than original MIMIC-IV source tables. Live notebook displays use in-memory results; saved-output replay and validation workflows may read these files.

**Live input behavior:** Normal execution requires the configured real BigQuery inputs and approved access. Missing settings, credentials, permissions or tables stop execution with a cause and a next step; no synthetic/demo data or saved results are substituted. Stored presentation-replay outputs are review artifacts, not a successful new data load. Use a fresh kernel for a new run.


In [ ]:
# Presentation uses only objects produced by this notebook execution.
from pathlib import Path as _ResultPath
import sys as _result_sys
_result_root = _ResultPath.cwd().resolve()
if _result_root.name == "notebooks":
    _result_root = _result_root.parent
if not (_result_root / "scripts" / "nonicu_notebook_results.py").is_file():
    raise RuntimeError("Run this notebook from the repository root or its notebooks directory")
if str(_result_root / "scripts") not in _result_sys.path:
    _result_sys.path.insert(0, str(_result_root / "scripts"))
from nonicu_notebook_results import NotebookResults
notebook_results = NotebookResults(globals())


In [ ]:
from pathlib import Path
import sys,json,hashlib,importlib.metadata as metadata
from datetime import datetime,timezone
import pandas as pd
from IPython.display import display
project_root=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(project_root/'scripts'))
from prediction_time_validation import evaluate,export_results,BOOTSTRAPS,SPLIT_SEED,BOOTSTRAP_SEED
from real_data_access import read_prediction_inputs as read_inputs
started_utc=datetime.now(timezone.utc).isoformat()


In [ ]:
PROJECT_ID = "mimic-iv-portfolio"
DATASET = "nonicu_raas"
# Reproducers must use their own authorized project/dataset; see REPRODUCIBILITY.md.
# The helper performs read-only SELECTs, a dry run and a 10 GiB/query cap.


In [ ]:
# Shared source references and execution mode for all numbered results.
notebook_results.overview(['Table 04c.1', 'Table 04c.2', 'Table 04c.3', 'Figure 04c.1'])


## 2. Load private inputs and check SQL/Python eligibility

In [ ]:
with notebook_results.capture('Table 04c.1', globals()):
    prediction_frame,query_checks=read_inputs(project_root,PROJECT_ID,DATASET)
    print('Adult base admissions loaded in memory:',len(prediction_frame))
    _notebook_result_value = (display(pd.DataFrame(query_checks['sql_python_eligibility_checks'])[['prediction_hours','admissions','deaths','bytes_processed']]))


## 3. Fit prespecified patient-disjoint models

In [ ]:
prediction_results=evaluate(prediction_frame,n_boot=BOOTSTRAPS,progress=lambda name:print('MODEL_COMPLETED',name))


## 4. Cohort flow and exclusion reasons

In [ ]:
with notebook_results.capture('Table 04c.2', globals()):
    display(prediction_results['cohorts'][['prediction_hours','eligible_admissions','eligible_patients','deaths','train_admissions','test_admissions','overlapping_patients','later_icu_admissions_retained','deaths_at_or_before_24h_retained']])
    display(prediction_results['exclusions'])
    _notebook_result_value = (print('Order-time coverage and source checks:',prediction_results['quality']))


## 5. Held-out discrimination and calibration

In [ ]:
with notebook_results.capture('Table 04c.3', globals()):
    display(prediction_results['metrics'][['model','test_admissions','test_deaths','roc_auc','roc_auc_CI_lower','roc_auc_CI_upper','pr_auc_average_precision','pr_auc_CI_lower','pr_auc_CI_upper','brier']].round(6))
    _notebook_result_value = (display(prediction_results['metrics'][['model','mean_predicted_risk','observed_test_risk','calibration_intercept','calibration_slope','calibration_fit_converged','iterations','bootstrap_successful']].round(6)))


## 6. Export aggregates and figures

In [ ]:
with notebook_results.capture('Figure 04c.1', globals()):
    export_results(prediction_results,project_root)
    _notebook_result_value = (print('Saved aggregate prediction-time tables and calibration figure; no record-level exports.'))


## 7. Execution receipt

In [ ]:
receipt={
    'started_utc':started_utc,'completed_utc':datetime.now(timezone.utc).isoformat(),
    'design':'Additional admission/24h internal prediction evaluation; independent of original 04b cohort/estimates',
    'outcome':'Death strictly after prediction time and no later than recorded hospital discharge; variable hospital-stay horizon',
    'split_seed':SPLIT_SEED,'bootstrap_seed_base':BOOTSTRAP_SEED,'bootstrap_requested':BOOTSTRAPS,
    'bootstrap_scope':'Held-out patients; fixed trained models; no model-training uncertainty',
    'queries':query_checks,'quality':prediction_results['quality'],
    'models':json.loads(prediction_results['metrics'].to_json(orient='records')),
    'cohorts':json.loads(prediction_results['cohorts'].to_json(orient='records')),
    'packages':{p:metadata.version(p) for p in ['pandas','numpy','scikit-learn','statsmodels']},
    'source_sha256':{p:hashlib.sha256((project_root/p).read_bytes()).hexdigest() for p in ['sql/05_prediction_time_inputs.sql','scripts/prediction_time_validation.py','docs/PREDICTION_TIME_PROTOCOL.md']},
    'row_exports':False,'prospective_or_external_validation':False,
}
receipt_path=project_root/'python/outputs/prediction_time_execution.json'
_ = receipt_path.write_text(json.dumps(receipt,indent=2,allow_nan=False)+'\n')
print('Saved aggregate-only execution receipt.')


## Interpretation limits

Patient separation and explicit prediction times improve the evaluation design, but these are still retrospective, limited-feature internal models. Demographics are assumed available at admission; original field-entry/version histories are unavailable. POE time bounds the companion feature's recorded order, not confirmed administration or immutable historical contents. Unverified/later order times never make the feature positive, and zero does not establish absence of medication. Calibration is diagnosed on the holdout without recalibrating predictions. Bootstrap intervals condition on fitted training models. No causal treatment effect, prospective clinical utility or external validity is established.
